In [1]:
import numpy as np

# Creation of reduced structure

**Workflow: Nanoribbon**
1. Create electrode: `build_electrode(w, l)`
    * width : number of atoms in *y* (image: 9)
    * length : how many *armchair* in *x* (image: 2)
2. Create center region: `build_nanoribbon(electrode, c)`
    * `electrode`: sisl object
    * center size : how many `electrodes` to use for center region.
    * automatically adds the left/right electrodes

<img src="ribbon_creation.png" width="800">



**Workflow: Reduced device** `build_reduced_device(ribbon, repeat=3)`

3. Change `electrode` atoms
    * left and right has different atoms
4. Copy structure, rotate around center and add to final structure
    * Repeat 3 times for angles : $\{0^\circ,\ \pm 60^\circ\}$ 

<img src="add_arms.png" width="800">

5. Remove atoms overlapping
    * If atoms has same position (tolerance: $0.1$ Å)
6. Save atom indecies for electrodes (left: N, right: O)
7. Change all atoms to C


## Compatibility issues with $w,\ l,\ c$
**Case**: Center size too small -- uncomment line 9

In [2]:
# using '__' to avoid overwriting the compatible parameters
from utils.structure import build_electrode, build_nanoribbon, build_reduced_device
from utils.plots import mark_electrode
w, l, c = 13, 2, 2
e = build_electrode(w, l)
r = build_nanoribbon(e, c)
d, lr_idx = build_reduced_device(r, e, repeat=3)
atoms_style = []
# atoms_style = mark_electrode(lr_idx) # uncomment to mark electrodes
d.plot(axes="xy", atoms_style=atoms_style) # plot device in xy plane

       Initial number of atoms: 624
 Atoms after removing overlaps: 372


## Infer dimensions

**Ribbon**: Geometric center close to high symmetry point (carbon ring center) $W = 5+4i$  
* If $W < 5$ geometric center is (likely) on a dimer.  
* Center region has to satisfy : $2(i+1) = 0 \mod{L}\rightarrow C = \frac{2(i+1)}{L}$
* Spacing between valid $i$: $s = \frac{L}{\gcd{(2,L)}}$  
* Lowest valid: $T = -1 \mod{s}$  
* Given $i'$ the next valid is $i = i' + (T-i')\mod{s}$  
Implemented in `utils.infer_centersize(l,i)`

In [ ]:
from utils import build_electrode, build_nanoribbon, infer_centersize
from utils.plots import plot_with_center, mark_electrode
l, i = 2, 5
l, w, c = infer_centersize(l, i)
# l,w,c = 2, 5, 2 --- IGNORE ---
print(f"Length: {l}, Width: {w}, Center size: {c}")

e = build_electrode(w, l) # width, length
r = build_nanoribbon(e, c) # electrode, center_size
d,lr_idxs = build_reduced_device(r, e, repeat=3) # device, electrode, amount of rotations
plot_with_center(r)

Length: 2, Width: 25, Center size: 6
       Initial number of atoms: 2400
 Atoms after removing overlaps: 1464


In [4]:
a_style = mark_electrode((lr_idxs))
plot_with_center(d, atoms_style=a_style)

### Why does 'compatible' dimensions matter?


In [7]:
import sisl
from sisl.viz.processors.math import normalize
SCALE = 0.6
OFFSET = 0.1
def plot_pdos_old(lwc):
    PARAMS = "{}L_{}W_{}C".format(*lwc)
    # extract data
    DATA = np.load(f"../notebooks/calculations/device_ldos_{PARAMS}.npz")
    ldos, lr_idx = DATA["ldos"], DATA["lr_idx"]
    # build device
    device = sisl.io.get_sile(f"../notebooks/structures/device_{PARAMS}.xyz").read_geometry()
    # scale pdos for plotting
    norm_pdos_avg_E = normalize(ldos.mean(axis=(0,1)))*SCALE + OFFSET
    style = mark_electrode(lr_idx)
    style.append({"size": norm_pdos_avg_E})
    return device.plot(axes="xy", atoms_style=style)

#### The bad case
Edge states in center region

In [8]:
lwc = 1, 5, 3
plot_pdos_old(lwc)

### If we choose *LWC* compatible

Still edge states in electrodes

In [11]:
lwc = infer_centersize(1, i=4)
plot_pdos_old(lwc)

### Does electrode length remove edge states?

In [78]:
lwc = infer_centersize(3, i=2)
plot_pdos_old(lwc)

## What Influence has this on the LDOS(E)

In [48]:
import inspect
device_docstr = inspect.getdoc(build_reduced_device)
print(device_docstr)

Build the reduced structure device along with indicis of left and right electrodes for each nanoribbon arm

Parameters
----------
nanoribbon : Geometry
    Ribbon to repeat, rotate and add to the final structure
electrode : Geometry
    The structure that is used for both left and right electrodes
repeat : int, optional
    Determines how many "arms" the structure should have, by default 3

Returns
-------
Geometry
    The device 
tuple[ndarray, ndarray]
    arrays of atomic indices for left and right electrodes each of shape (repeat, N)

Raises
------
ValueError
    If `repeat` is not either 1, 2, or 3


In [ ]:
from utils.energies import multi_LDOS
ldos_docstr = inspect.getdoc(multi_LDOS)
print(ldos_docstr)
# ldos_args = inspect.getcomments(multi_LDOS)
# print(ldos_args)